# Capa Silver – Limpieza, tipado, imputación y segmentación · Consumo Alumbrado Público (Sep 2026)

Transforma las grillas crudas de `workspace.fenomeno_nino_bronze` en tablas tipadas y **segmentadas por sección del reporte** en `workspace.fenomeno_nino_silver`. Reglas de la capa:

* Trazabilidad total: cada tabla conserva `fila_origen` (fila del Excel) y `fecha_procesamiento`.
* Imputación **documentada**: nunca se inventan valores; cada campo faltante queda en NULL (o 0 si es la convención del archivo) con nota en `nota_imputacion` o en la tabla `supuestos_imputacion`.
* Los subtotales y filas TOTAL se conservan con banderas (`es_subtotal`, `tipo_registro`) para no perder los controles del archivo.

## 1. Lectura de la capa Bronze

Silver **no vuelve a leer el Excel**: consume las grillas crudas de Bronze (lineage Bronze → Silver → Gold). Las celdas vacías llegan como `""` y se normalizan a NaN.

In [0]:
import pandas as pd
import numpy as np
from pyspark.sql import functions as F

CATALOGO = "workspace"
SCHEMA_BRONZE = "fenomeno_nino_bronze"
SCHEMA_SILVER = "fenomeno_nino_silver"

def cargar_grid(nombre_tabla):
    """Lee una grilla cruda de Bronze → pandas, con fila_excel como índice y NaN en celdas vacías."""
    g = (spark.table(f"{CATALOGO}.{SCHEMA_BRONZE}.{nombre_tabla}")
         .drop("archivo_origen", "fecha_ingesta")
         .toPandas()
         .replace("", np.nan)
         .set_index("fila_excel"))
    return g

def num(serie):
    """Convierte a numérico; texto no convertible (encabezados, notas) → NaN."""
    return pd.to_numeric(serie, errors="coerce")

g1 = cargar_grid("septiembre_2026_raw")       # filas 1–32, columnas c1–c9
g2 = cargar_grid("reduccion_horaria_raw")      # filas 1–46, columnas c1–c11
print("Bronze leída:", g1.shape, "y", g2.shape)

## 2. Tabla `parametros`

Filas 5–7 de ambas hojas. Los tres parámetros se repiten en las dos hojas: se **verifica consistencia** y se deduplica.

In [0]:
FILAS_PARAM = {5: "dias_del_mes", 6: "factor_stn2", 7: "valor_unitario_pesos_kwh"}
registros = []
for hoja, g in [("Septiembre 2026", g1), ("Reducción horaria", g2)]:
    for fila, nombre in FILAS_PARAM.items():
        fuente = g.loc[fila, "c4"]
        registros.append({
            "parametro": nombre,
            "valor": float(g.loc[fila, "c3"]),
            "fuente": None if pd.isna(fuente) else fuente,
            "hoja_origen": hoja,
        })
parametros = pd.DataFrame(registros)

# Consistencia entre hojas antes de deduplicar
chk = parametros.pivot_table(index="parametro", columns="hoja_origen", values="valor", aggfunc="first")
assert np.allclose(chk.iloc[:, 0], chk.iloc[:, 1]), f"Parámetros difieren entre hojas:\n{chk}"

parametros = parametros.drop_duplicates(subset=["parametro", "valor"]).sort_values("parametro").reset_index(drop=True)
display(parametros)

## 3. Tabla `ajustes_factura`

Filas 11–15 de la hoja *Septiembre 2026* (sección «AJUSTE DE ENERGÍA EN LA FACTURA»). Nota de limpieza: en la fila TOTAL, la posición `hoja_origen` contiene 131708 (potencia total W de canchas); se mueve a la columna auxiliar `potencia_w_aux` para no falsear el campo.

In [0]:
COLS_AJ = ["concepto", "hoja_origen", "puntos_luminosos", "kwh_mes_factura", "kwh_stn2", "valor_pesos", "observacion"]
aj = g1.loc[11:15, "c2":"c8"].copy()
aj.columns = COLS_AJ
aj = aj.reset_index().rename(columns={"fila_excel": "fila_origen"})
for c in ["puntos_luminosos", "kwh_mes_factura", "kwh_stn2", "valor_pesos"]:
    aj[c] = num(aj[c])

aj["tipo_registro"] = np.where(aj["concepto"].eq("TOTAL"), "total", "ajuste")
aj["potencia_w_aux"] = np.where(aj["tipo_registro"].eq("total"), num(aj["hoja_origen"]), np.nan)
aj.loc[aj["tipo_registro"].eq("total"), "hoja_origen"] = None
display(aj)

## 4. Tabla `consumo_referencia_puentes`

Filas 19–23 de la hoja *Septiembre 2026* (sección «CONSUMO DE REFERENCIA DE LOS PUENTES», encabezado en la fila 18): consumo propio de los puentes antes de descuentos, con potencia, cantidad, horas/día, kWh y valor.

In [0]:
COLS_RF = ["concepto", "origen", "potencia_w", "cantidad", "horas_dia", "kwh_mes", "kwh_stn2", "valor_pesos"]
rf = g1.loc[19:23, "c2":"c9"].copy()
rf.columns = COLS_RF
rf = rf.reset_index().rename(columns={"fila_excel": "fila_origen"})
for c in COLS_RF[2:]:
    rf[c] = num(rf[c])
rf["es_subtotal"] = rf["concepto"].str.startswith("Subtotal", na=False)
display(rf)

## 5. Tabla `consumo_escenarios` (con imputación documentada)

Filas 12–18 de la hoja *Reducción horaria* (sección «CONSUMO MENSUAL»). **Canchas y arquitectónicos** llegan sin horas/día (celdas amarillas del archivo): esos campos quedan en NULL con nota de imputación — no se inventan valores.

In [0]:
COLS_CE = ["concepto", "potencia_w", "cantidad", "carga_kw", "horas_dia_base", "kwh_base", "kwh_menos_1h", "kwh_menos_2h", "kwh_menos_3h", "kwh_menos_4h"]
ce = g2.loc[12:18, "c2":"c11"].copy()
ce.columns = COLS_CE
ce = ce.reset_index().rename(columns={"fila_excel": "fila_origen"})
for c in COLS_CE[1:]:
    ce[c] = num(ce[c])
ce["es_total"] = ce["concepto"].eq("TOTAL")
ce["es_subtotal"] = ce["concepto"].str.startswith("Subtotal", na=False)

# Imputación documentada: horas/día ausentes → NULL + nota (no se fabrica el valor)
mask_sin_horas = ce["horas_dia_base"].isna() & ~ce["es_total"] & ~ce["es_subtotal"]
ce["nota_imputacion"] = np.where(
    mask_sin_horas,
    "Horas/día no reportadas en el archivo (celda amarilla editable); pendiente supuesto del negocio — ver tabla supuestos_imputacion",
    None,
)
display(ce)

## 6. Tablas `ahorro_escenarios` y `valor_escenarios`

Filas 22–28 (ahorro kWh y %) y 32–38 (valor en $) de la hoja *Reducción horaria*. En canchas/arquitectónicos los vacíos se imputan a **0** por la convención explícita del archivo («sin potencia ni horas ⇒ sin ahorro»), con nota de trazabilidad.

In [0]:
COLS_AH = ["concepto", "ahorro_kwh_1h", "ahorro_kwh_2h", "ahorro_kwh_3h", "ahorro_kwh_4h", "pct_1h", "pct_2h", "pct_3h", "pct_4h"]
ah = g2.loc[22:28, "c2":"c10"].copy()
ah.columns = COLS_AH
ah = ah.reset_index().rename(columns={"fila_excel": "fila_origen"})
for c in COLS_AH[1:]:
    ah[c] = num(ah[c])
mask_conf = ah["concepto"].str.contains("por confirmar", na=False)
ah.loc[mask_conf, COLS_AH[1:]] = ah.loc[mask_conf, COLS_AH[1:]].fillna(0)
ah["es_total"] = ah["concepto"].eq("TOTAL")
ah["nota_imputacion"] = np.where(mask_conf, "0 por convención del archivo: sin potencia/horas reportadas no hay ahorro calculable", None)

COLS_VA = ["concepto", "valor_base", "valor_menos_1h", "valor_menos_2h", "valor_menos_3h", "valor_menos_4h", "ahorro_pesos_1h", "ahorro_pesos_2h", "ahorro_pesos_3h", "ahorro_pesos_4h"]
va = g2.loc[32:38, "c2":"c11"].copy()
va.columns = COLS_VA
va = va.reset_index().rename(columns={"fila_excel": "fila_origen"})
for c in COLS_VA[1:]:
    va[c] = num(va[c])
mask_conf_v = va["concepto"].str.contains("por confirmar", na=False)
va.loc[mask_conf_v, COLS_VA[1:]] = va.loc[mask_conf_v, COLS_VA[1:]].fillna(0)
va["es_total"] = va["concepto"].eq("TOTAL")
va["nota_imputacion"] = np.where(mask_conf_v, "0 por convención del archivo: sin potencia/horas reportadas no hay valor calculable", None)

display(ah)
display(va)

## 7. Tabla `supuestos_imputacion`

Registro editable de los supuestos pendientes o aplicados. Hoy contiene los campos que el archivo deja abiertos (celdas amarillas). Cuando el negocio fije las horas/día de canchas y arquitectónicos, se actualiza el supuesto aquí y se reprocesa.

In [0]:
supuestos = pd.DataFrame([
    {"tabla": "consumo_escenarios", "concepto": "Canchas (potencia y horas por confirmar)", "campo": "horas_dia_base",
     "valor_supuesto": np.nan, "estado": "pendiente",
     "justificacion": "El archivo trae kWh agregados de temporización sin horas de operación (celda amarilla editable)."},
    {"tabla": "consumo_escenarios", "concepto": "Arquitectónicos (potencia y horas por confirmar)", "campo": "horas_dia_base",
     "valor_supuesto": np.nan, "estado": "pendiente",
     "justificacion": "Ídem canchas: falta la horas/día base para calcular escenarios de reducción."},
    {"tabla": "ajustes_factura", "concepto": "Puente de la 4 Sur (descuento)", "campo": "kwh_mes_factura",
     "valor_supuesto": 0, "estado": "aplicado_archivo",
     "justificacion": "Días de descuento = 0 en septiembre según el propio archivo."},
    {"tabla": "ahorro_escenarios", "concepto": "Canchas/Arquitectónicos", "campo": "ahorro_kwh_*",
     "valor_supuesto": 0, "estado": "aplicado_archivo",
     "justificacion": "Sin potencia ni horas reportadas, el archivo asume ahorro 0; se imputa 0 con nota de trazabilidad."},
])
display(supuestos)

## 8. Guardado en la capa Silver

Escribe las 7 tablas en `workspace.fenomeno_nino_silver` con `fecha_procesamiento`. Los NULL de columnas de texto se normalizan antes de escribir.

In [0]:
def guardar(df_pd, nombre):
    df_limp = df_pd.copy()
    for c in df_limp.columns:
        if df_limp[c].dtype == object:
            df_limp[c] = df_limp[c].where(df_limp[c].notna(), None)
    df = spark.createDataFrame(df_limp).withColumn("fecha_procesamiento", F.current_timestamp())
    df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{CATALOGO}.{SCHEMA_SILVER}.{nombre}")
    print(f"{nombre:32} → {df.count()} filas")

parametros["fila_origen"] = None
for nombre, df_pd in [
    ("parametros", parametros),
    ("ajustes_factura", aj),
    ("consumo_referencia_puentes", rf),
    ("consumo_escenarios", ce),
    ("ahorro_escenarios", ah),
    ("valor_escenarios", va),
    ("supuestos_imputacion", supuestos),
]:
    guardar(df_pd, nombre)

## 9. Validación cruzada

Tres familias de controles:

1. **Totales del archivo**: cada fila TOTAL parseada debe coincidir con el valor visible en el Excel.
2. **Sumas internas**: la suma de conceptos debe reproducir la fila TOTAL (no puede haber pérdida de filas en el parseo).
3. **Coherencia entre tablas**: puntos luminosos, y valor base = kWh base × factor STN2 × tarifa.

In [0]:
resultados = []
def check(nombre, real, esperado, tol=1e-3):
    ok = abs(real - esperado) <= tol * max(1.0, abs(esperado))
    resultados.append({"control": nombre, "real": real, "esperado": esperado, "estado": "OK" if ok else "FALLA"})

# 1. Totales del archivo (fila TOTAL parseada vs valor del Excel)
check("Ajuste TOTAL kWh factura", aj.loc[aj.tipo_registro == "total", "kwh_mes_factura"].iloc[0], -194188.402294)
check("Ajuste TOTAL $ factura", aj.loc[aj.tipo_registro == "total", "valor_pesos"].iloc[0], -129903999.623912)
check("Consumo TOTAL kWh base", ce.loc[ce.es_total, "kwh_base"].iloc[0], 6886.5)
check("Valor TOTAL $ base", va.loc[va.es_total, "valor_base"].iloc[0], 4606783.324032)
check("Ahorro TOTAL $ -4h", va.loc[va.es_total, "ahorro_pesos_4h"].iloc[0], 3499991.338319)
check("Ahorro TOTAL % -4h", ah.loc[ah.es_total, "pct_4h"].iloc[0], 0.759747)
check("Ref. puentes 4 Sur 50 W (kWh)", rf.iloc[0]["kwh_mes"], 4312.5)

# 2. Sumas internas: conceptos = fila TOTAL
check("Σ ajustes = TOTAL kWh", aj.loc[aj.tipo_registro == "ajuste", "kwh_mes_factura"].sum(), -194188.402294)
check("Σ puntos luminosos = TOTAL", aj.loc[aj.tipo_registro == "ajuste", "puntos_luminosos"].sum(), 4209)
check("Σ puentes 4 Sur = Subtotal kWh", rf.loc[~rf.es_subtotal & rf["concepto"].str.startswith("Puente 4"), "kwh_mes"].sum(), 6292.5)

# 3. Coherencia entre tablas: valor = kWh × factor × tarifa
factor = parametros.loc[parametros.parametro == "factor_stn2", "valor"].iloc[0]
tarifa = parametros.loc[parametros.parametro == "valor_unitario_pesos_kwh", "valor"].iloc[0]
kwh_total = ce.loc[ce.es_total, "kwh_base"].iloc[0]
check("Valor TOTAL $ = kWh × factor × tarifa", kwh_total * factor * tarifa, va.loc[va.es_total, "valor_base"].iloc[0], tol=1e-4)

df_check = pd.DataFrame(resultados)
display(df_check)
assert (df_check["estado"] == "OK").all(), f"Controles fallidos:\n{df_check[df_check.estado != 'OK']}"
print(f"\n{len(resultados)} controles OK — Silver validada.")